# 통합 합성 데이터 가설 검증 실험

실제 데이터 분석에서 도출된 세 가지 핵심 발견을 합성 데이터로 검증한다:
- **H1**: 개인화된 기저선 추적의 중요성 (OLS vs Mixed)
- **H2**: 다중 지표 통합 모니터링 우위 (부하 단독 vs 부하+HRV)
- **H3**: Monotony 독립 효과 및 억제변수 재현
- **H4**: 결측 민감도 분석 (100회 Monte Carlo)

ADR-012 참조.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt

from src.data.synthetic_integrated import (
    DEFAULT_PARAMS, generate_integrated_dataset, inject_missingness,
)
from src.stats.mixed_effects import (
    fit_random_intercept, extract_model_metrics,
)
from src.stats.cross_validation import loso_cv, loso_summary

PARAMS = DEFAULT_PARAMS.copy()
print(f"설정: {PARAMS['n_athletes']}선수 x {PARAMS['n_days']}일, seed={PARAMS['seed']}")

## 0. 데이터 생성

In [ ]:
df_complete, random_effects = generate_integrated_dataset(PARAMS, return_complete=True)

df_valid = df_complete.dropna(subset=[
    "acwr_ra", "mono", "strain_val", "hooper_next", "ln_rmssd_next",
]).copy().reset_index(drop=True)

print(f"총 행수: {len(df_complete)}, 유효 관측수: {len(df_valid)}")
df_valid.describe()

## H1: 개인화된 기저선 추적의 중요성

In [ ]:
# OLS vs Mixed — Hooper
ols_hooper = sm.OLS.from_formula(
    "hooper_next ~ acwr_ra + mono + strain_val", data=df_valid
).fit()

mixed_hooper = fit_random_intercept(
    "hooper_next ~ acwr_ra + mono + strain_val", df_valid, "athlete"
)

y_true = df_valid["hooper_next"].values
ss_tot = np.sum((y_true - y_true.mean()) ** 2)
mixed_r2 = 1.0 - np.sum((y_true - mixed_hooper.fittedvalues.values) ** 2) / ss_tot

print(f"OLS  — AIC: {ols_hooper.aic:.1f}, R²: {ols_hooper.rsquared:.4f}")
print(f"Mixed — AIC: {extract_model_metrics(mixed_hooper, df_valid, 'hooper_next')['aic']:.1f}, R²: {mixed_r2:.4f}")
print(f"ΔAIC: {ols_hooper.aic - extract_model_metrics(mixed_hooper, df_valid, 'hooper_next')['aic']:.1f}")

In [ ]:
# OLS vs Mixed — HRV
ols_hrv = sm.OLS.from_formula(
    "ln_rmssd_next ~ acwr_ra + mono", data=df_valid
).fit()

mixed_hrv = fit_random_intercept(
    "ln_rmssd_next ~ acwr_ra + mono", df_valid, "athlete"
)

y_true_h = df_valid["ln_rmssd_next"].values
ss_tot_h = np.sum((y_true_h - y_true_h.mean()) ** 2)
mixed_r2_h = 1.0 - np.sum((y_true_h - mixed_hrv.fittedvalues.values) ** 2) / ss_tot_h

print(f"OLS  — AIC: {ols_hrv.aic:.1f}, R²: {ols_hrv.rsquared:.4f}")
print(f"Mixed — AIC: {extract_model_metrics(mixed_hrv, df_valid, 'ln_rmssd_next')['aic']:.1f}, R²: {mixed_r2_h:.4f}")

## H2: 다중 지표 통합 모니터링 우위

In [ ]:
m_load = fit_random_intercept(
    "hooper_next ~ acwr_ra + mono + strain_val", df_valid, "athlete"
)
m_int = fit_random_intercept(
    "hooper_next ~ acwr_ra + mono + strain_val + ln_rmssd_next", df_valid, "athlete"
)

r2_load = 1.0 - np.sum((y_true - m_load.fittedvalues.values) ** 2) / ss_tot
r2_int = 1.0 - np.sum((y_true - m_int.fittedvalues.values) ** 2) / ss_tot
f2 = (r2_int - r2_load) / (1.0 - r2_int) if r2_int < 1.0 else np.nan

m_load_aic = extract_model_metrics(m_load, df_valid, "hooper_next")["aic"]
m_int_aic = extract_model_metrics(m_int, df_valid, "hooper_next")["aic"]

print(f"M_load AIC: {m_load_aic:.1f}, R²: {r2_load:.4f}")
print(f"M_integrated AIC: {m_int_aic:.1f}, R²: {r2_int:.4f}")
print(f"ΔAIC: {m_load_aic - m_int_aic:.1f}")
print(f"증분 Cohen's f²: {f2:.4f}")

## H3: Monotony 독립 효과 및 억제변수 재현

In [ ]:
m1 = fit_random_intercept("hooper_next ~ acwr_ra", df_valid, "athlete")
m2 = fit_random_intercept("hooper_next ~ acwr_ra + mono", df_valid, "athlete")
m3 = fit_random_intercept("hooper_next ~ acwr_ra + mono + strain_val", df_valid, "athlete")
m4 = fit_random_intercept("hooper_next ~ acwr_ew + mono + strain_val", df_valid, "athlete")

print("순차 투입 Monotony 계수:")
print(f"  Step2 (+Mono):    β={m2.fe_params.get('mono', np.nan):.4f}")
print(f"  Step3 (+Strain):  β={m3.fe_params.get('mono', np.nan):.4f}, p={m3.pvalues.get('mono', np.nan):.4f}")
print(f"  Step4 (EWMA):     β={m4.fe_params.get('mono', np.nan):.4f}")
print(f"  참값: {PARAMS['beta_mono_hooper']}")

## H4: 결측 민감도 분석 (100회 Monte Carlo)

In [ ]:
N_MC = 100
TRUE_MONO = PARAMS["beta_mono_hooper"]
MECHANISMS = ["complete", "mcar", "mar", "mnar"]

mc_results = {m: {"beta_mono": []} for m in MECHANISMS}

for i in range(N_MC):
    iter_params = PARAMS.copy()
    iter_params["seed"] = PARAMS["seed"] + i
    df_iter = generate_integrated_dataset(iter_params)
    df_iter_v = df_iter.dropna(subset=["acwr_ra", "mono", "strain_val", "hooper_next"]).copy()

    for mech in MECHANISMS:
        if mech == "complete":
            df_a = df_iter_v
        else:
            mc_rng = np.random.default_rng(PARAMS["seed"] * 1000 + i)
            df_a = inject_missingness(df_iter_v, mc_rng, mech, PARAMS)
            df_a = df_a.dropna(subset=["hooper_next"])

        if len(df_a) < 100:
            continue
        try:
            model = smf.mixedlm(
                "hooper_next ~ acwr_ra + mono + strain_val",
                data=df_a, groups=df_a["athlete"],
            )
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                result = model.fit(reml=False)
            mc_results[mech]["beta_mono"].append(result.fe_params.get("mono", np.nan))
        except Exception:
            continue

    if (i + 1) % 25 == 0:
        print(f"진행: {i + 1}/{N_MC}")

print("\n결과:")
for mech in MECHANISMS:
    betas = np.array(mc_results[mech]["beta_mono"])
    bias = np.nanmean(betas) - TRUE_MONO
    print(f"  {mech:>10}: mean={np.nanmean(betas):.5f}, bias={bias:.5f}, rel_bias={abs(bias)/abs(TRUE_MONO):.2%}")

## 종합 결과

각 가설별 Pass/Fail 판정은 `notebooks/run_integrated_hypothesis.py` 스크립트 실행 결과를 참조한다.
보고서: `reports/integrated_hypothesis_report.md`